# 03 - Flujo agentic simple, sin caja negra

Este notebook muestra un patrón agentic sin framework: planner, writer y reviewer. Cada etapa muestra `system_prompt`, `user_prompt`, `messages`, raw output y objeto Pydantic cuando aplica. Después se muestra el equivalente con `PromptRegistry`.

## Qué NO estamos usando todavía

- No Agents SDK.
- No LangGraph.
- No WebSearchTool.
- No RAG.

Release 01 solo necesita entender el cableado básico entre roles LLM.

In [35]:
import base64
import json
from urllib.parse import parse_qs, quote_plus, urlparse

from playwright.async_api import Error as PlaywrightError, async_playwright

from IPython.display import Markdown, display

from llmkit.config.settings import get_settings
from llmkit.llms import LLMFactory, build_messages
from llmkit.prompts import PromptRegistry
from llmkit.schemas import ReportData, ReportReview, WebSearchPlan, parse_json_output

get_settings.cache_clear()
settings = get_settings()

QUESTION = "What AI agent frameworks should an industrial analytics team evaluate in 2026?"
MODEL_ID = settings.default_model
RUN_LIVE_LLM_CALLS = True

print("Question:", QUESTION)
print("Model:", MODEL_ID)

Question: What AI agent frameworks should an industrial analytics team evaluate in 2026?
Model: gpt-5-nano


## Etapa 1: planner manual

El planner no investiga. Solo propone búsquedas o líneas de investigación. Su salida se valida con `WebSearchPlan`.

In [36]:
planner_system_prompt = (
    "You are a research planner. Given a research question, produce only valid JSON with a list of search tasks. "
    "Each task needs a reason and a query. Do not perform the research yourself."
)

planner_user_prompt = (
    f"Create a research plan for this question:\n\n{QUESTION}\n\n"
    "Return JSON with this shape:\n"
    '{"searches": [{"reason": "...", "query": "..."}]}'
)

planner_messages = build_messages(planner_system_prompt, planner_user_prompt)
print("SYSTEM:\n", planner_system_prompt)
print("\nUSER:\n", planner_user_prompt)
print("\nMESSAGES:\n", planner_messages)

SYSTEM:
 You are a research planner. Given a research question, produce only valid JSON with a list of search tasks. Each task needs a reason and a query. Do not perform the research yourself.

USER:
 Create a research plan for this question:

What AI agent frameworks should an industrial analytics team evaluate in 2026?

Return JSON with this shape:
{"searches": [{"reason": "...", "query": "..."}]}

MESSAGES:
 [{'role': 'system', 'content': 'You are a research planner. Given a research question, produce only valid JSON with a list of search tasks. Each task needs a reason and a query. Do not perform the research yourself.'}, {'role': 'user', 'content': 'Create a research plan for this question:\n\nWhat AI agent frameworks should an industrial analytics team evaluate in 2026?\n\nReturn JSON with this shape:\n{"searches": [{"reason": "...", "query": "..."}]}'}]


In [37]:
if RUN_LIVE_LLM_CALLS:
    llm = LLMFactory.create(MODEL_ID)
    planner_response = llm.invoke(
        system=planner_system_prompt,
        user=planner_user_prompt,
        response_format={"type": "json_object"},
    )
    raw_plan_json = planner_response.content
else:
    raw_plan_json = json.dumps({
        "searches": [
            {"reason": "Compare orchestration frameworks", "query": "LangGraph CrewAI AutoGen agent frameworks industrial analytics"},
            {"reason": "Check production concerns", "query": "AI agent framework observability evaluation deployment"},
            {"reason": "Identify practical business use cases", "query": "agentic AI workflows business use cases 2026"},
        ]
    })

research_plan = parse_json_output(raw_plan_json, WebSearchPlan)
print("RAW PLAN JSON:\n", raw_plan_json)
research_plan

RAW PLAN JSON:
 {
  "searches": [
    {
      "reason": "Establish the current landscape of AI agent frameworks relevant to industrial analytics and manufacturing as of 2026, to identify candidates to evaluate.",
      "query": "2026 landscape of AI agent frameworks for manufacturing and industrial analytics"
    },
    {
      "reason": "Compare popular agent-building toolkits (e.g., LangChain, Auto-GPT, ReAct) and assess fit for industrial use cases.",
      "query": "LangChain Auto-GPT ReAct agent frameworks comparison 2026 industrial analytics"
    },
    {
      "reason": "Identify open-source vs commercial options to balance flexibility, support, and risk for an industrial analytics team.",
      "query": "open-source vs commercial AI agent frameworks 2026 industrial"
    },
    {
      "reason": "Assess interoperability with industrial data protocols and systems (OPC UA, MQTT, ISA-95, MES, ERP).",
      "query": "AI agent frameworks OPC UA MQTT integration 2026"
    },
    {
   

WebSearchPlan(searches=[WebSearchItem(reason='Establish the current landscape of AI agent frameworks relevant to industrial analytics and manufacturing as of 2026, to identify candidates to evaluate.', query='2026 landscape of AI agent frameworks for manufacturing and industrial analytics'), WebSearchItem(reason='Compare popular agent-building toolkits (e.g., LangChain, Auto-GPT, ReAct) and assess fit for industrial use cases.', query='LangChain Auto-GPT ReAct agent frameworks comparison 2026 industrial analytics'), WebSearchItem(reason='Identify open-source vs commercial options to balance flexibility, support, and risk for an industrial analytics team.', query='open-source vs commercial AI agent frameworks 2026 industrial'), WebSearchItem(reason='Assess interoperability with industrial data protocols and systems (OPC UA, MQTT, ISA-95, MES, ERP).', query='AI agent frameworks OPC UA MQTT integration 2026'), WebSearchItem(reason='Evaluate deployment models (cloud, edge, hybrid) and late

## Etapa 2: herramienta de búsqueda gratuita

Esta etapa ejecuta el plan con Playwright y Chromium headless sobre Bing Search. No usa WebSearchTool ni APIs pagadas.

In [38]:
def decode_bing_url(result_url: str) -> str:
    parsed = urlparse(result_url)
    if "bing.com" not in parsed.netloc or parsed.path != "/ck/a":
        return result_url

    encoded_url = parse_qs(parsed.query).get("u", [""])[0]
    if not encoded_url.startswith("a1"):
        return result_url

    payload = encoded_url[2:]
    padding = "=" * (-len(payload) % 4)
    return base64.urlsafe_b64decode(payload + padding).decode("utf-8")


async def free_web_search(browser, query: str, max_results: int = 3) -> list[dict[str, str]]:
    url = f"https://www.bing.com/search?q={quote_plus(query)}&cc=US&setlang=en-US"
    page = await browser.new_page(
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    )
    try:
        for attempt in range(2):
            try:
                await page.goto(url, wait_until="commit", timeout=30_000)
                await page.wait_for_load_state("domcontentloaded", timeout=10_000)
                break
            except PlaywrightError:
                if attempt == 1:
                    raise
                await page.wait_for_timeout(1_000)

        links = page.locator("li.b_algo h2 a")
        count = min(await links.count(), max_results)
        results = []
        for index in range(count):
            link = links.nth(index)
            title = ((await link.text_content()) or "").strip()
            result_url = decode_bing_url(await link.get_attribute("href") or "")
            if not title or not result_url:
                continue
            results.append({"title": title, "url": result_url})
        return results
    except PlaywrightError as exc:
        return [{"title": f"Search failed: {exc}", "url": url}]
    finally:
        await page.close()


async with async_playwright() as playwright:
    browser = await playwright.chromium.launch(headless=True)
    search_results = []
    for search in research_plan.searches:
        search_results.append({
            "reason": search.reason,
            "query": search.query,
            "results": await free_web_search(browser, search.query),
        })
    await browser.close()

research_notes = "\n".join(
    f"- {item['reason']} ({item['query']}): "
    + "; ".join(f"{result['title']} - {result['url']}" for result in item["results"])
    for item in search_results
)

print(research_notes)

- Establish the current landscape of AI agent frameworks relevant to industrial analytics and manufacturing as of 2026, to identify candidates to evaluate. (2026 landscape of AI agent frameworks for manufacturing and industrial analytics): 2026 - Wikipedia - https://en.wikipedia.org/wiki/2026; Year 2026 Calendar – United States - timeanddate.com - https://www.timeanddate.com/calendar/?year=2026&country=1; 2026 Calendar - https://www.calendar-365.com/2026-calendar.html
- Compare popular agent-building toolkits (e.g., LangChain, Auto-GPT, ReAct) and assess fit for industrial use cases. (LangChain Auto-GPT ReAct agent frameworks comparison 2026 industrial analytics): LangChain: Observe, Evaluate, and Deploy Reliable AI Agents - https://www.langchain.com/; GitHub - langchain-ai/langchain: The agent engineering platform - https://github.com/langchain-ai/langchain; Introduction to LangChain - GeeksforGeeks - https://www.geeksforgeeks.org/artificial-intelligence/introduction-to-langchain/
- I

## Etapa 3: writer manual

El writer recibe la pregunta y las notas. No planifica ni busca. Solo sintetiza el reporte final.

In [39]:
writer_system_prompt = (
    "You are a senior research writer. You receive a question and prepared notes. "
    "Synthesize them into a clear markdown report with a short summary, structured body, and follow-up questions."
)

writer_user_prompt = (
    f"Original question:\n{QUESTION}\n\n"
    f"Prepared research notes:\n{research_notes}\n\n"
    "Write a markdown report and include follow-up questions."
)

writer_messages = build_messages(writer_system_prompt, writer_user_prompt)
print("SYSTEM:\n", writer_system_prompt)
print("\nUSER PREVIEW:\n", writer_user_prompt[:1200])
print("\nMESSAGES:\n", writer_messages)

SYSTEM:
 You are a senior research writer. You receive a question and prepared notes. Synthesize them into a clear markdown report with a short summary, structured body, and follow-up questions.

USER PREVIEW:
 Original question:
What AI agent frameworks should an industrial analytics team evaluate in 2026?

Prepared research notes:
- Establish the current landscape of AI agent frameworks relevant to industrial analytics and manufacturing as of 2026, to identify candidates to evaluate. (2026 landscape of AI agent frameworks for manufacturing and industrial analytics): 2026 - Wikipedia - https://en.wikipedia.org/wiki/2026; Year 2026 Calendar – United States - timeanddate.com - https://www.timeanddate.com/calendar/?year=2026&country=1; 2026 Calendar - https://www.calendar-365.com/2026-calendar.html
- Compare popular agent-building toolkits (e.g., LangChain, Auto-GPT, ReAct) and assess fit for industrial use cases. (LangChain Auto-GPT ReAct agent frameworks comparison 2026 industrial anal

In [40]:
if RUN_LIVE_LLM_CALLS:
    llm = LLMFactory.create(MODEL_ID)
    writer_response = llm.invoke(system=writer_system_prompt, user=writer_user_prompt)
    markdown_report = writer_response.content
else:
    markdown_report = """# Agent Frameworks for Industrial Analytics

Industrial analytics teams should compare agent frameworks by workflow control, observability, evaluation support, and deployment risk.

## Practical comparison

LangGraph is a strong candidate when workflows need explicit state, routing, retries, and checkpoints. CrewAI can be useful for role-based prototypes and business demos. AutoGen remains useful for multi-agent conversation experiments.

## Recommendation

Start with a simple LLM pipeline, add evaluation data, then test LangGraph for workflows that truly need stateful orchestration.

## Follow-up questions

- Which workflow should be prototyped first?
- What production monitoring is required?
"""

report = ReportData(
    short_summary="Compare frameworks by control, observability, and deployment risk.",
    markdown_report=markdown_report,
    follow_up_questions=["Which workflow should be prototyped first?", "What production monitoring is required?"],
)

display(Markdown(report.markdown_report))

# Evaluating AI Agent Frameworks for Industrial Analytics in 2026

Summary
Industrial analytics teams should evaluate a mix of AI agent frameworks that support robust tool-use, reasoning, and integration with industrial data ecosystems. The primary candidates often cited are LangChain, Auto-GPT, and ReAct-style approaches. Key evaluation dimensions include interoperability with OPC UA, MQTT, ISA-95, MES/ERP; deployment options (cloud, edge, hybrid); security and governance; observability and safety; and total cost of ownership. A structured PoC program across digital twin/simulation workflows, real-time factory data, and platform observability is recommended to inform adoption.

Structured report

1) 2026 landscape overview
- The field continues to evolve toward practical, industrial-grade AI agents that can orchestrate tools, query data sources, and interact with manufacturing systems.
- Open-source ecosystems (e.g., LangChain) offer flexibility, rapid iteration, and strong tooling around prompt management, chains, and observability. Commercial options can provide broader support, compliance assurances, and enterprise-grade governance.
- Interoperability with industrial data protocols (OPC UA, MQTT) and enterprise systems (ISA-95 workflows, MES/ERP) remains a critical gating factor for production environments.
- Adoption planning should consider deployment models (cloud, edge, hybrid) and the associated latency, reliability, and security implications.

2) Candidate frameworks to evaluate (fit for industrial use cases)
- LangChain
  - Strengths: mature agent tooling, chains/orchestrations, plug-ins for various tools, strong ecosystem and documentation.
  - Industrial fit: favorable for building end-to-end workflows that coordinate multiple data sources and tools; good observability and tooling around prompt management.
- Auto-GPT
  - Strengths: emphasis on autonomous task execution and loop closure, capable of multi-step task orchestration.
  - Industrial fit: useful for autonomous agents that must run end-to-end tasks, but requires careful governance to avoid runaway loops and data leakage; robust monitoring and safeguards are essential.
- ReAct (Reasoning + Acting) approaches
  - Strengths: explicit separation of reasoning and action steps, effective for tasks requiring deliberate planning and tool use.
  - Industrial fit: pairs well with systems that require transparent decision traces and controllable actions; benefits from solid state management and auditing.
- Practical notes
  - In practice, many industrial teams adopt a hybrid: LangChain as the backbone with ReAct-style prompting patterns, and Auto-GPT-like orchestration where appropriate, all under strong governance and observability.
- References: official LangChain docs and GitHub, general introductions to LangChain, and contemporary discussions of agent design patterns.

3) Open-source vs commercial options
- Open-source considerations
  - Pros: flexibility, transparency, customizable governance, lower upfront licensing friction; easier to tailor for proprietary industrial data pipelines.
  - Cons: requires in-house capability to maintain security, compliance, and long-term support; potential variability in support quality.
- Commercial considerations
  - Pros: vendor-supported SLAs, enterprise governance features, integrated security/compliance offerings, predictable roadmaps.
  - Cons: potential licensing costs, dependence on vendor for critical updates, risk of lock-in.
- Actionable takeaway: define a 3–5 year total cost of ownership (TCO) and governance model early, and consider a staged approach with a core open-source platform augmented by commercial security/compliance layers or support for critical lines of business.

4) Interoperability with industrial data protocols and systems
- Critical data interfaces include OPC UA, MQTT; ERP/MES/ISA-95 workflows; data historians and PLC interfaces.
- Evaluation plan should include:
  - Existence and quality of adapters/connectors for OPC UA, MQTT, and common MES/ERP sources.
  - Ability to ingest real-time streaming data and historical data with proper time synchronization.
  - Data governance and lineage support for industrial data.
- Practical implication: prioritize frameworks and toolchains with proven or easily implementable connectors to core shop-floor systems.

5) Deployment models, latency, and reliability
- Deployment considerations:
  - Edge deployment for low-latency, local decision-making and data sovereignty.
  - Cloud deployment for scalability, centralized governance, and advanced model workloads.
  - Hybrid configurations combining edge inference with cloud augmentation.
- Latency/reliability expectations:
  - Real-time or near-real-time analytics may require sub-second to a few seconds latency; high-throughput scheduling or optimization tasks may tolerate higher latencies but require reliability guarantees.
- Action: design PoCs that measure end-to-end latency, uptime, and resilience under network partition or data loss scenarios.

6) Security, safety, governance, and compliance
- Core concerns: access control, data handling and leakage prevention, prompt-injection and adversarial risk, model versioning, and change management.
- Best practices to test during evaluation:
  - Role-based access and least-privilege policies for agents and tools.
  - Model and prompt governance, auditing, and rollback procedures.
  - Compliance mapping to industry standards (e.g., security best practices like CompTIA Security+ guidance).
- Practical takeaway: security and governance should be a first-class criterion in PoCs, not after-the-fact.

7) Integration with digital twin, simulation, and optimization workflows
- Value proposition: AI agents that can ingest digital twin data, run simulations, and optimize plant-level parameters or schedules.
- What to verify:
  - Compatibility with existing digital twins and simulation toolchains.
  - Ability to parameterize optimization problems, test scenarios, and report on outcomes with traceable rationale.
- Rationale: this integration often drives the most measurable ROI in manufacturing analytics.

8) Evaluation criteria and metrics
- Task effectiveness metrics:
  - Task success rate, completion time, and throughput.
- Reasoning and planning metrics:
  - Quality of reasoning trace, justification clarity, and auditable decision logs.
- Robustness and reliability:
  - Behavior under noisy/partial data, fault tolerance, and graceful degradation.
- Explainability and safety:
  - Transparency of tool use, prompt-safety checks, and user override capability.
- Observability and maintainability:
  - Instrumentation coverage, error budgets, and debuggability.
- Data/control risk:
  - Data leakage, model drift, and access controls validation.
- Practical note: align metrics with specific industrial use cases (e.g., predictive maintenance, production scheduling, anomaly detection).

9) Real-world case studies, pilots, and benchmarks
- The field has ongoing pilots and pilots in manufacturing and industrial contexts; however, comprehensive public case studies are still emerging.
- Action item for teams: conduct targeted literature and vendor-sponsored pilots in areas like scheduling optimization, anomaly detection, or robotic automation within a controlled scope.
- Possible reference anchors: industrial AI pilots, digital twin integrations, and general AI agent benchmarking efforts. Cross-check with industry case studies and vendor roadmaps for concrete examples.

10) Observability, debugging, and tool-usage safety
- Observability stack should cover:
  - Logs, metrics, traces for agent actions, tool calls, and data access.
  - Prompt/toolkits management, versioning, and rollback capabilities.
- Safety focus:
  - Guardrails to prevent unsafe tool usage, data exfiltration, or uncontrolled actions.
  - Clear audit trails for all agent decisions and tool invocations.

11) Vendor roadmaps and long-term planning
- Consider not only current capabilities but also:
  - Roadmaps for model updates, tool integrations, and security/compliance enhancements.
  - Enterprise support commitments and upgrade cycles.
- Action: map internal adoption plans to vendor timelines and plan PoCs in alignment with 12–24 month horizons.

12) Total cost of ownership and budgeting
- Direct costs: framework licenses (if any), hosting, edge devices, data pipeline infrastructure.
- Indirect costs: engineering time for integration, governance overhead, security/ops staffing.
- Recommendation: develop a budget envelope for a staged adoption (pilot, scale, and sustain phases) and incorporate ongoing maintenance.

13) Next steps (practical plan)
- Assemble a cross-functional evaluation team including data scientists, industrial IT/OT leads, security/compliance, and manufacturing operations.
- Define 2–3 manufacturing use cases to pilot (e.g., anomaly detection with real-time alerts, production scheduling optimization, digital twin-informed decision support).
- Develop a shortlisting framework and scoring rubric based on the evaluation criteria above.
- Run 2–3 PoCs in parallel or sequentially, with clear success criteria and exit criteria.
- Establish a governance model for data access, model management, and incident response.

Follow-up questions
1) Which 2–3 manufacturing use cases are highest priority for an initial PoC (e.g., predictive maintenance, anomaly detection, scheduling/optimization, quality control)?
2) What is the current data architecture depth for OPC UA, MQTT, MES/ERP, and historians, and how easily can adapters be added?
3) What are the required latency targets for on-floor decision-making, and what is acceptable latency for cloud-based analytics?
4) Is there an internal capability to manage and audit prompts, tool inventories, and model versions, or is vendor support required for governance?
5) What are the preferred deployment models (edge-only, cloud-only, or hybrid) given plant locations, data sovereignty, and IT/OT policies?
6) What level of security/compliance assurance is required (SOC 2, ISO 27001, industry-specific standards), and who is responsible for ongoing security audits?
7) Are there existing digital twin or simulation platforms in use, and how well do they integrate with AI agent workflows?
8) What is the acceptable risk tolerance for autonomous agent actions and potential loops or failures, and how will override/rollback mechanisms be implemented?
9) How will success be measured beyond technical metrics (operational impact, ROI, maintenance reduction, downtime)?
10) What is the preferred timeline for pilots, approvals, and budget cycles for 2026–2027?
11) Do you prefer open-source-first with commercial security overlays, or a fully vendor-provided solution with integrated governance?
12) Which data governance, data quality, and lineage requirements must be met before moving from PoC to production?

References / sources (illustrative, based on notes)
- LangChain official resources and ecosystem
  - LangChain official site
  - LangChain GitHub repository
  - GeeksforGeeks introduction to LangChain
- ReAct/agent design patterns and industrial applicability
- Observability and governance resources
  - IBM on observability
  - Dynatrace on observability
- Security and governance references
  - CompTIA Security+ (security governance best practices)
- Digital twin and AI in manufacturing context
  - MIT News AI topics and related machine intelligence coverage
- General evaluation and methodology references
  - Meera Michigan Mech/Evaluation basics (educational material)
  
Note: The synthesis above draws on the provided notes, with a focus on a practical, deployment-oriented evaluation framework suitable for manufacturing environments. Where possible, consult the latest vendor documentation, industry case studies, and product roadmaps to refine the shortlist and PoC plans.

## Etapa 4: reviewer manual

El reviewer evalúa la salida con JSON validado. Esto prepara el terreno para evaluadores automáticos en releases posteriores, pero aquí sigue siendo explícito.

In [41]:
reviewer_system_prompt = (
    "You are a strict report reviewer. Evaluate whether a markdown report answers the original question, "
    "uses the provided notes, and leaves clear next steps. Return only valid JSON matching the requested schema. "
    "The score must be a decimal from 0.0 to 1.0, not a percentage or a 1-100 score."
)

reviewer_user_prompt = (
    f"Original question:\n{QUESTION}\n\n"
    f"Markdown report:\n{report.markdown_report}\n\n"
    "Return JSON with keys score, passed, feedback, and follow_up_questions. "
    "Use score as a decimal from 0.0 to 1.0. Example: use 0.92, not 92."
)

reviewer_messages = build_messages(reviewer_system_prompt, reviewer_user_prompt)
print("SYSTEM:\n", reviewer_system_prompt)
print("\nUSER PREVIEW:\n", reviewer_user_prompt[:1200])
print("\nMESSAGES:\n", reviewer_messages)

SYSTEM:
 You are a strict report reviewer. Evaluate whether a markdown report answers the original question, uses the provided notes, and leaves clear next steps. Return only valid JSON matching the requested schema. The score must be a decimal from 0.0 to 1.0, not a percentage or a 1-100 score.

USER PREVIEW:
 Original question:
What AI agent frameworks should an industrial analytics team evaluate in 2026?

Markdown report:
# Evaluating AI Agent Frameworks for Industrial Analytics in 2026

Summary
Industrial analytics teams should evaluate a mix of AI agent frameworks that support robust tool-use, reasoning, and integration with industrial data ecosystems. The primary candidates often cited are LangChain, Auto-GPT, and ReAct-style approaches. Key evaluation dimensions include interoperability with OPC UA, MQTT, ISA-95, MES/ERP; deployment options (cloud, edge, hybrid); security and governance; observability and safety; and total cost of ownership. A structured PoC program across digit

In [42]:
if RUN_LIVE_LLM_CALLS:
    llm = LLMFactory.create(MODEL_ID)
    review_response = llm.invoke(
        system=reviewer_system_prompt,
        user=reviewer_user_prompt,
        response_format={"type": "json_object"},
    )
    raw_review_json = review_response.content
else:
    raw_review_json = json.dumps({
        "score": 0.82,
        "passed": True,
        "feedback": "Good first synthesis. Add concrete evaluation criteria before using it for vendor selection.",
        "follow_up_questions": ["Which framework integrates best with monitoring?", "What evaluation set should be created first?"]
    })
review = parse_json_output(raw_review_json, ReportReview)
print("RAW REVIEW JSON:\n", raw_review_json)
review

RAW REVIEW JSON:
 {
  "score": 0.92,
  "passed": true,
  "feedback": "The markdown report provides a comprehensive evaluation framework for AI agent frameworks suitable for industrial analytics in 2026. It identifies LangChain, Auto-GPT, and ReAct-style approaches as candidates, discusses interoperability with OPC UA, MQTT, ISA-95, MES/ERP, deployment models, security/governance, observability, and PoC planning, and includes a practical next steps section plus well-formed follow-up questions. It aligns well with the user's question and uses the provided notes to justify criteria and actions. The report also codifies a structured evaluation plan (sections 4-12) and concrete next steps (section 13). The inclusion of real-world constraints (edge vs cloud, latency, data governance) makes it actionable.\n\nAreas for improvement: Consider widening the candidate set to include emerging or domain-specific frameworks and components (e.g., LangChain-derived tool-agnostic orchestration layers, in

ReportReview(score=0.92, passed=True, feedback="The markdown report provides a comprehensive evaluation framework for AI agent frameworks suitable for industrial analytics in 2026. It identifies LangChain, Auto-GPT, and ReAct-style approaches as candidates, discusses interoperability with OPC UA, MQTT, ISA-95, MES/ERP, deployment models, security/governance, observability, and PoC planning, and includes a practical next steps section plus well-formed follow-up questions. It aligns well with the user's question and uses the provided notes to justify criteria and actions. The report also codifies a structured evaluation plan (sections 4-12) and concrete next steps (section 13). The inclusion of real-world constraints (edge vs cloud, latency, data governance) makes it actionable.\n\nAreas for improvement: Consider widening the candidate set to include emerging or domain-specific frameworks and components (e.g., LangChain-derived tool-agnostic orchestration layers, integration with industr

## Variante: contrato local con BaseModel

La misma idea puede definirse directamente en el notebook con `BaseModel`. Esto es util cuando el contrato de salida todavia esta en exploracion y no merece vivir en `llmkit.schemas`.

In [43]:
from pydantic import BaseModel, Field

class EvaluationDimension(BaseModel):
    name: str = Field(description="Dimension being evaluated.")
    score: float = Field(ge=1, le=5, description="Score from 1 to 5.")
    weight: float = Field(ge=0, le=1, description="Relative importance from 0 to 1.")
    evidence: str = Field(description="Short textual evidence supporting the score.")

class ResearchQualityEval(BaseModel):
    verdict: str = Field(description="Short final judgement.")
    dimensions: list[EvaluationDimension]
    weighted_score: float = Field(ge=1, le=5)

base_model_prompt = (
    "Return only valid JSON matching this Pydantic schema. "
    "Evaluate clarity, factuality, and actionability."
)
print(ResearchQualityEval.model_json_schema())

{'$defs': {'EvaluationDimension': {'properties': {'name': {'description': 'Dimension being evaluated.', 'title': 'Name', 'type': 'string'}, 'score': {'description': 'Score from 1 to 5.', 'maximum': 5, 'minimum': 1, 'title': 'Score', 'type': 'number'}, 'weight': {'description': 'Relative importance from 0 to 1.', 'maximum': 1, 'minimum': 0, 'title': 'Weight', 'type': 'number'}, 'evidence': {'description': 'Short textual evidence supporting the score.', 'title': 'Evidence', 'type': 'string'}}, 'required': ['name', 'score', 'weight', 'evidence'], 'title': 'EvaluationDimension', 'type': 'object'}}, 'properties': {'verdict': {'description': 'Short final judgement.', 'title': 'Verdict', 'type': 'string'}, 'dimensions': {'items': {'$ref': '#/$defs/EvaluationDimension'}, 'title': 'Dimensions', 'type': 'array'}, 'weighted_score': {'maximum': 5, 'minimum': 1, 'title': 'Weighted Score', 'type': 'number'}}, 'required': ['verdict', 'dimensions', 'weighted_score'], 'title': 'ResearchQualityEval', 't

In [45]:
if RUN_LIVE_LLM_CALLS:
    llm = LLMFactory.create(MODEL_ID)
    base_model_response = llm.invoke(
        system=base_model_prompt,
        user=f"Evaluate this report:\n\n{report.markdown_report}",
        response_format={"type": "json_object"},
    )
    raw_quality_json = base_model_response.content
else:
    raw_quality_json = json.dumps({
        "verdict": "Useful first pass, but needs sharper production criteria.",
        "dimensions": [
            {"name": "clarity", "score": 4.0, "weight": 0.30, "evidence": "The recommendation is easy to follow."},
            {"name": "factuality", "score": 3.5, "weight": 0.45, "evidence": "It stays within the prepared notes."},
            {"name": "actionability", "score": 3.0, "weight": 0.25, "evidence": "It names next steps but not acceptance criteria."},
        ],
        "weighted_score": 3.55,
    })

quality_eval = parse_json_output(raw_quality_json, ResearchQualityEval)
print("RAW QUALITY JSON:\n", raw_quality_json)
quality_eval

StructuredOutputError: Output does not match schema ResearchQualityEval. Raw output: {
  "report_title": "Evaluating AI Agent Frameworks for Industrial Analytics in 2026",
  "evaluation_date": "2026-04-27",
  "overall_assessment": {
    "rating_out_of_5": 4.0,
    "summary": "The report is clear, practically oriented, and deployment-focused. It covers critical dimensions (interoperability, deployment models, security, governance, observability) and provides a coherent PoC plan. It could be strengthened with more concrete, measurable PoC success criteria, a standardized scoring rubric, and explicit ROI-linked metrics to facilitate faster decision-making."
  },
  "strengths": [
    "Structured, deployment-oriented framing that aligns with industrial realities (OPC UA, MQTT, ISA-95, MES/ERP).",
    "Balanced discussion of open-source and commercial options with governance and observability emphasis.",
    "Practical PoC guidance and a comprehensive follow-up questions section to tailor ev

## Equivalente toolkit

El registry solo guarda los prompts que acabamos de escribir manualmente. No convierte esto en LangGraph ni Agents SDK. Solo evita repetir strings.

In [ ]:
planner_prompt = PromptRegistry.get("research.plan")
writer_prompt = PromptRegistry.get("research.report")
reviewer_prompt = PromptRegistry.get("research.review")

toolkit_planner_user = planner_prompt.render_user(question=QUESTION)
toolkit_planner_messages = build_messages(planner_prompt.system, toolkit_planner_user)

print("Registry prompt:", planner_prompt.name)
print("Equivalent messages:")
toolkit_planner_messages

Registry prompt: research.plan
Equivalent messages:


[{'role': 'system',
  'content': 'You are a research planner. Given a research question, produce only valid JSON with a list of search tasks. Each task needs a reason and a query. Do not perform the research yourself.'},
 {'role': 'user',
  'content': 'Create a research plan for this question:\n\nWhat AI agent frameworks should an industrial analytics team evaluate in 2026?\n\nReturn JSON with this shape:\n{"searches": [{"reason": "...", "query": "..."}]}'}]

## Mapa mental

| Manual | Toolkit Release 01 | Posible Release 03 |
| --- | --- | --- |
| `system_prompt` + `user_prompt` | `PromptTemplate` | Agent instructions |
| `messages` | `build_messages()` | Framework message state |
| `llm.invoke(...)` | `LLMFactory` + normalized response | Model node |
| `parse_json_output(...)` | Pydantic schemas | Router/evaluator node |

Primero entendemos el cableado. Después decidimos si vale la pena convertirlo en grafo.